# Clean Teaching Notebook: DPO for a Vision-Language Model (SmolVLM-Instruct)

This notebook is a **teaching-first rewrite** of the Hugging Face cookbook notebook for fine-tuning **`HuggingFaceTB/SmolVLM-Instruct`** with **DPO** on the **`HuggingFaceH4/rlaif-v_formatted`** preference dataset.

It is designed to help you:
1. understand **what each step does** and **why it is needed**,
2. run the workflow on **Google Colab**,
3. adapt it into **your own notebook** later.

---

## How to create this notebook locally

### Option A — easiest
1. Save this file as `smolvlm_dpo_teaching_notebook.ipynb`.
2. Open it with:
   - **Jupyter Notebook**
   - **JupyterLab**
   - **VS Code** with the Jupyter extension

### Option B — from scratch
1. Create a new folder, for example:
   ```bash
   mkdir smolvlm_dpo_teaching
   cd smolvlm_dpo_teaching
   ```
2. Create a Python environment if you want to run locally:
   ```bash
   python -m venv .venv
   source .venv/bin/activate   # Linux / macOS
   # or
   .venv\Scripts\activate    # Windows
   ```
3. Install Jupyter:
   ```bash
   pip install notebook jupyterlab
   ```
4. Put this `.ipynb` file in the folder and open it.

---

## How to put it on Colab

### Option A — upload the notebook
1. Go to **Google Colab**
2. Click **File -> Upload notebook**
3. Upload this `.ipynb` file

### Option B — store it in Google Drive
1. Put the notebook into your Google Drive
2. In Colab, open it from Drive

### GPU recommendation
For your first run:
- **L4**: best match to the original cookbook
- **A100**: safest and most forgiving
- **T4**: may work for lighter experiments, but you may need to reduce settings
- **H100**: excellent, but usually more than you need for this teaching notebook

---

## Main references
- Hugging Face cookbook notebook:
  `https://huggingface.co/learn/cookbook/en/fine_tuning_vlm_dpo_smolvlm_instruct`
- Hugging Face blog on DPO for VLMs:
  `https://huggingface.co/blog/dpo_vlm`
- SmolVLM-Instruct model card:
  `https://huggingface.co/HuggingFaceTB/SmolVLM-Instruct`

## 1. High-level idea: practice DPO as one example of RLHF

This notebook uses **Direct Preference Optimization (DPO)** as a practical entry point into **RLHF / preference alignment**.

### Big picture
We start from a pretrained **vision-language model** (VLM) that can read:
- text,
- images,
- or both together,

and produce **text outputs**.

We then train it using **preference data**. Each example contains:
- a **prompt**,
- an **image**,
- a **chosen** answer,
- a **rejected** answer.

So instead of saying:

> "Here is the one exact correct label."

we say:

> "Between these two candidate answers, this one is preferred."

That is the key intuition behind preference optimization.

### Why this is related to RLHF
In the broader RLHF pipeline, people often talk about:
1. supervised fine-tuning,
2. reward modeling,
3. policy optimization with RL such as PPO.

DPO is attractive because it uses **preference pairs directly**, and in practice it often provides a simpler alignment path than a full reward-model-plus-RL pipeline.

### Conceptual DPO goal
For the same input \(x\), we want the model to give a higher score to the chosen answer than to the rejected answer:

\[
\log \pi_\theta(y_{\text{chosen}} \mid x)
>
\log \pi_\theta(y_{\text{rejected}} \mid x)
\]

### What we will do in this notebook
1. install the libraries,
2. load and inspect the dataset,
3. inspect the base VLM,
4. estimate whether it fits on the current GPU,
5. run baseline inference before alignment,
6. explain why naive full fine-tuning is hard,
7. make training feasible using:
   - lower precision / quantization,
   - LoRA / PEFT,
   - gradient checkpointing,
   - gradient accumulation,
8. fine-tune with DPO,
9. compare pre-alignment and post-alignment behavior.

### Important note
This notebook is written as a **teaching notebook**, not a minimal benchmark script.  
That means some cells are intentionally verbose and inspect shapes, memory, and sample structure.

## 2. Install libraries

We need a small stack of libraries:

- **transformers**: model classes, processor, generation, quantization hooks
- **trl**: DPO training utilities such as `DPOTrainer` and `DPOConfig`
- **datasets**: loading and preprocessing Hugging Face datasets
- **bitsandbytes**: 4-bit / 8-bit quantization for memory-efficient loading
- **peft**: LoRA / DoRA adapters for parameter-efficient fine-tuning
- **accelerate**: helps with device placement and training setup
- **flash-attn**: faster and more memory-efficient attention kernels on supported GPUs
- **Pillow**: image manipulation
- **tensorboard**: optional logging

If you run this notebook on Colab, install these first, then restart the runtime if needed.

In [ ]:
!pip install -U -q transformers trl datasets bitsandbytes peft accelerate pillow tensorboard
!pip install -q flash-attn --no-build-isolation

## 3. Load the dataset, convert images to RGB, and inspect its structure

The dataset used in the cookbook is:

`HuggingFaceH4/rlaif-v_formatted`

Each sample is a **preference example** for a VLM.  
Conceptually, each row contains:
- a **prompt**
- one or more **images**
- a **chosen** response
- a **rejected** response

Why each part is needed:
- **prompt**: the user question / instruction
- **images**: the visual context
- **chosen**: the preferred answer
- **rejected**: the less preferred answer

DPO needs **relative preference information**, so we need both `chosen` and `rejected`.

We also convert all images to **RGB** so the image processor sees a consistent format.

In [ ]:
from datasets import load_dataset
from PIL import Image
from pprint import pprint

dataset_id = "HuggingFaceH4/rlaif-v_formatted"
train_dataset, test_dataset = load_dataset(dataset_id, split=["train[:2%]", "test[:1%]"])

def ensure_rgb(example):
    imgs = example.get("images", None)
    if imgs is None or len(imgs) == 0:
        return example
    image = imgs[0]
    if isinstance(image, Image.Image) and image.mode != "RGB":
        image = image.convert("RGB")
    example["images"] = [image]
    return example

train_dataset = train_dataset.map(ensure_rgb, num_proc=2)
test_dataset = test_dataset.map(ensure_rgb, num_proc=2)

print("Train size:", len(train_dataset))
print("Test size :", len(test_dataset))
print("\\nDataset columns:")
print(train_dataset.column_names)

sample = train_dataset[0]
print("\\nSample keys:")
print(sample.keys())

print("\\nPrompt example:")
pprint(sample["prompt"])

print("\\nChosen example:")
pprint(sample["chosen"])

print("\\nRejected example:")
pprint(sample["rejected"])

print("\\nNumber of images in sample:", len(sample["images"]))
print("First image type:", type(sample["images"][0]))
print("First image mode:", sample["images"][0].mode)
print("First image size:", sample["images"][0].size)

sample["images"][0]

## 4. Load the VLM and inspect its architecture and metadata

We use:

`HuggingFaceTB/SmolVLM-Instruct`

Key points from the model card:
- it is a **multimodal image+text model**
- it is based on **Idefics3**
- it uses **SmolLM2** as the text backbone
- it uses a **shape-optimized SigLIP** image encoder
- it uses **81 visual tokens per 384×384 image patch**
- the model card lists **2B params** and **BF16** tensor type
- the processor default uses `N=4`, which corresponds to an image longest edge of **1536 px** because \(4 \times 384 = 1536\)

Why this matters:
- the vision side determines image preprocessing cost,
- the language side determines text generation behavior,
- total parameter count and precision strongly affect VRAM use.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

model_id = "HuggingFaceTB/SmolVLM-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
base_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=base_dtype,
    _attn_implementation="flash_attention_2" if torch.cuda.is_available() else "eager",
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model id:", model_id)
print("Device:", device)
print("Base dtype:", base_dtype)
print("Total params:", f"{total_params:,}")
print("Trainable params before PEFT:", f"{trainable_params:,}")

print("\\n=== Model config summary ===")
print(model.config)

print("\\n=== Processor summary ===")
print(processor)

if hasattr(processor, "image_processor") and hasattr(processor.image_processor, "size"):
    print("\\nImage processor size config:", processor.image_processor.size)

## 5. Inspect the current GPU and compare it with rough memory needs

This cell helps answer:

> "What GPU am I on, how much VRAM do I have, and is this setup likely to fit?"

Important distinction:
- **inference memory** is much smaller than full training memory,
- **DPO training** needs extra memory for:
  - model,
  - gradients,
  - optimizer state,
  - activations,
  - and conceptually a reference behavior in DPO-style training.

A rough parameter-memory formula is:

\[
\text{weight memory} \approx N \times P
\]

where:
- \(N\) = number of parameters
- \(P\) = bytes per parameter

In [ ]:
def bytes_per_param(dtype_name: str):
    table = {
        "fp32": 4.0,
        "float32": 4.0,
        "fp16": 2.0,
        "float16": 2.0,
        "bf16": 2.0,
        "bfloat16": 2.0,
        "int8": 1.0,
        "4bit": 0.5,
    }
    return table[dtype_name]

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    total_vram_gb = props.total_memory / (1024**3)
    print("GPU:", gpu_name)
    print(f"Total VRAM: {total_vram_gb:.2f} GB")
else:
    print("No CUDA GPU detected.")

print("\\nApproximate parameter-memory only (weights only):")
for dtype_name in ["fp32", "bf16", "int8", "4bit"]:
    mem_gb = total_params * bytes_per_param(dtype_name) / (1024**3)
    print(f"{dtype_name:>5}: ~{mem_gb:.2f} GB")

print("\\nReminder:")
print("- This does NOT include activations, KV cache, gradients, optimizer states, or dataloader overhead.")
print("- Full fine-tuning needs much more than just the weight memory.")
print("- Parameter-efficient tuning is the main reason this notebook can run on smaller GPUs.")

## 6. First baseline run: can the base model do inference before any alignment?

Before training, we should run the model **as-is** on a sample.

This is important because:
1. we want a **baseline**,
2. we want to verify the processor and model pipeline work,
3. later we will compare pre-alignment and post-alignment behavior.

Important teaching point:
- **Base inference may work**
- but **naive full fine-tuning may still be impractical**

So this cell is not here to prove that the model cannot do inference.
It shows that the pretrained model already works, while training is a different memory/computation problem.

In [ ]:
from pprint import pprint

@torch.no_grad()
def generate_text_from_sample(model, processor, sample, max_new_tokens=128, device="cuda"):
    text_input = processor.apply_chat_template(
        sample["prompt"],
        add_generation_prompt=True,
        tokenize=False,
    )

    image = sample["images"][0]
    if image.mode != "RGB":
        image = image.convert("RGB")

    inputs = processor(
        text=text_input,
        images=[image],
        return_tensors="pt",
    ).to(device)

    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    trimmed_generated_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        trimmed_generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    return output_text[0], inputs

baseline_sample = test_dataset[0]
baseline_output, baseline_inputs = generate_text_from_sample(
    model, processor, baseline_sample, max_new_tokens=128, device=device
)

print("=== Prompt ===")
pprint(baseline_sample["prompt"])

print("\\n=== Chosen ===")
pprint(baseline_sample["chosen"])

print("\\n=== Rejected ===")
pprint(baseline_sample["rejected"])

print("\\n=== Base model output ===")
print(baseline_output)

print("\\n=== Input tensor shapes ===")
for k, v in baseline_inputs.items():
    if hasattr(v, "shape"):
        print(k, tuple(v.shape), v.dtype)

## 7. Why naive full fine-tuning is hard, and the techniques used to make it work

### Why naive full fine-tuning is hard
If you update **all** parameters, you usually need memory for:
- model weights,
- gradients,
- optimizer states,
- activations,
- and training overhead.

### The techniques we use

#### A. Quantization
Quantization reduces the precision used to store model weights.

Examples:
- fp32: 4 bytes / parameter
- bf16 or fp16: 2 bytes / parameter
- int8: 1 byte / parameter
- int4: 0.5 byte / parameter

In Hugging Face, the common entry point is `BitsAndBytesConfig`.

Main parameters used here:
- `load_in_4bit=True`
- `bnb_4bit_use_double_quant=True`
- `bnb_4bit_quant_type="nf4"`
- `bnb_4bit_compute_dtype=torch.bfloat16`

#### B. LoRA
LoRA = **Low-Rank Adaptation**.

Instead of updating the whole weight matrix \(W\), LoRA adds a low-rank trainable update:

\[
W' = W + BA
\]

where:
- \(W\) is frozen,
- \(A\) and \(B\) are small trainable matrices.

Typical LoRA parameters:
- `r`: rank
- `lora_alpha`: scaling factor
- `lora_dropout`: dropout on the LoRA path
- `target_modules`: which layers receive adapters

#### C. Gradient checkpointing
Reduces activation memory by recomputing some activations during backprop.

#### D. Gradient accumulation
Simulates a larger effective batch:

\[
\text{effective batch size}
=
\text{per-device batch size}
\times
\text{gradient accumulation steps}
\]

In [ ]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

peft_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=["down_proj", "o_proj", "k_proj", "q_proj", "gate_proj", "up_proj", "v_proj"],
    use_dora=True,
    init_lora_weights="gaussian",
)

print("BitsAndBytesConfig:")
print(bnb_config)

print("\\nLoRA config:")
print(peft_config)

## 8. Reload the model in a memory-efficient way and measure trainable parameters

Now we switch from the large baseline model object to a training-ready setup:
- quantized base model,
- LoRA / DoRA adapters,
- very small trainable fraction.

In [ ]:
import gc
from peft import get_peft_model

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

del model
clear_memory()

model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    _attn_implementation="flash_attention_2" if torch.cuda.is_available() else "eager",
)

processor = AutoProcessor.from_pretrained(model_id)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 9. Evaluate the memory-efficient model before DPO

Even though we added LoRA adapters, they are still **untrained** at this point.

So this cell gives a pre-DPO baseline for the exact training setup we will use.

In [ ]:
pre_dpo_sample = test_dataset[1]
pre_dpo_output, _ = generate_text_from_sample(
    model, processor, pre_dpo_sample, max_new_tokens=128, device=device
)

print("=== Prompt ===")
pprint(pre_dpo_sample["prompt"])

print("\\n=== Chosen ===")
pprint(pre_dpo_sample["chosen"])

print("\\n=== Rejected ===")
pprint(pre_dpo_sample["rejected"])

print("\\n=== Output BEFORE DPO ===")
print(pre_dpo_output)

## 10. Align the model with DPO

This is the actual alignment stage.

### Why these training arguments look small
Large VLMs are memory-heavy, so the cookbook uses:
- `per_device_train_batch_size=1`
- `gradient_accumulation_steps=32`
- `gradient_checkpointing=True`

This teaching notebook uses smaller debug values first so it is easier to run.

In [ ]:
from trl import DPOConfig, DPOTrainer

output_dir = "smolvlm_dpo_teaching_output"

training_args = DPOConfig(
    output_dir=output_dir,
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    dataset_num_proc=2,
    dataloader_num_workers=2,
    logging_steps=5,
    report_to="none",
    save_strategy="steps",
    save_steps=20,
    save_total_limit=1,
    eval_steps=20,
    eval_strategy="steps",
    push_to_hub=False,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=processor,
    peft_config=peft_config,
)

# Uncomment when ready:
# trainer.train()

print("Trainer is ready.")
print("Uncomment trainer.train() when you are ready to run alignment.")

## 11. Re-test on the same data after alignment

After training, compare outputs on the **same evaluation sample**.

That gives you a controlled before/after comparison.

In [ ]:
# Uncomment after training:
# trainer.save_model(output_dir)

try:
    post_dpo_output, _ = generate_text_from_sample(
        model, processor, pre_dpo_sample, max_new_tokens=128, device=device
    )

    print("=== SAME PROMPT ===")
    pprint(pre_dpo_sample["prompt"])

    print("\\n=== CHOSEN ===")
    pprint(pre_dpo_sample["chosen"])

    print("\\n=== REJECTED ===")
    pprint(pre_dpo_sample["rejected"])

    print("\\n=== OUTPUT AFTER DPO (or current adapter state) ===")
    print(post_dpo_output)

    print("\\n=== COMPARE ===")
    print("Before DPO:\\n", pre_dpo_output)
    print("\\nAfter DPO:\\n", post_dpo_output)

except Exception as e:
    print("You probably have not run training yet, or the model state is unchanged.")
    print("Error:", repr(e))

## 12. Extra comments and suggested improvements

A few additions I recommend beyond the original outline:

### A. Add a compact config cell near the top
Keep together:
- `model_id`
- `dataset_id`
- subset sizes
- `max_new_tokens`
- output directory
- LoRA rank
- batch size
- accumulation steps
- number of epochs

### B. Add a memory-debug cell
Print:
- allocated GPU memory
- reserved GPU memory
- GPU name
- dtype
- parameter count

### C. Add a qualitative evaluation loop
Evaluate 3–5 fixed samples:
- prompt
- chosen
- rejected
- output before DPO
- output after DPO

### D. Be careful with the phrase "show it cannot work as is"
The accurate version is:
- **inference can often work as-is**
- **full fine-tuning may not fit or may be inefficient**
- therefore we use quantization + LoRA + checkpointing + accumulation

That is better teaching and more honest.

### E. Good Colab progression
1. Run baseline inference on **L4 or A100**
2. Run a tiny DPO debug training
3. Save the adapter
4. Compare outputs
5. Then scale subset size / epochs